# **Sea Level Pressure (SLP) Feature Tracking using SCIPY**

## Overview

Within this notebook, we will cover:
1. How to filter continuous SLP data do identify features of interest.
1. How to identify the lowest SLP minima associated with ETCs.
2. How to use SCIPY's "label" function to identify coherent objects within the SLP field
1. How to track identified ETCs through successive time steps.

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Intro to Cartopy](https://foundations.projectpythia.org/core/cartopy/cartopy) | Helpful | |
| [Understanding of NetCDF](https://foundations.projectpythia.org/core/data-formats/netcdf-cf) | Necessary | Familiarity with metadata structure |
| [Intro to Numpy](https://foundations.projectpythia.org/core/numpy/) | Necessary | Basic n-dimensional array indexing
| Project management | Helpful | |

## Imports

In [ ]:
import xarray as xr
import numpy as np
from scipy.ndimage import minimum_filter
from scipy.ndimage import label, generate_binary_structure
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import math

## Core functions for this notebook

### Distance function

This function serves to calculate the distance between two given latitudes, and longitudes in degrees.

In [ ]:
def distance(lon1, lat1, lon2, lat2):
        radius = 6371
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2) * math.sin(dlon/2)
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
        d = radius * c
        return d


### Crosses dateline function

This function ensures that features adjecent to each other, but on the international dateline, are considered as the same feature.

In [ ]:
def crosses_dateline(labeled_arr, resolution):
    #Shift 180 degrees
    shifted_arr = np.roll(labeled_arr, 180*resolution, axis = 1)

    #Examine a 3x3 region centered at the dateline
    for lat in range(0, len(shifted_arr) - 1):
    	sample_arr = shifted_arr[lat:lat+2, 179:181]

    	top_left = sample_arr[0, 0]
    	top_right = sample_arr[0, 1]
    	bottom_right = sample_arr[1, 1]
    	bottom_left = sample_arr[1, 0]
		
    	#Compare top left with position next to it
    	if(top_left != 0 and top_right != 0):
    		if(top_left != top_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]

    	#Compare top left with position diagonal to it
    	if(top_left != 0 and bottom_right != 0):
    		if(top_left != bottom_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
		
    	#Compate bottom left with position next to it
    	if(bottom_left != 0 and bottom_right != 0):
    		if(bottom_left != bottom_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    			#print('Changed arr bot left, bot right', sample_arr)
    	
    	#Compare bottom left with position diagonal to it
    	if(bottom_left != 0 and top_right != 0):
    		if(bottom_left != top_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    		
    return np.roll(shifted_arr, 180, axis = 1)
    
    

## Read in and visualize our raw data

For our purposes, we will look at the ERA5 datasets provided by **Copernicus**. This data provides an accurate and seamless record of Earth’s atmosphere, utilizing computer models and global observations. 

The easiest way to download their data is to navigate to [Climate Data Store](https://cds.climate.copernicus.eu/datasets?kw=Spatial+coverage%3A+Global&kw=Variable+domain%3A+Land+%28hydrology%29) and filter to the type of data you want to work with. For this notebook, the dataset is already provided. 

In [ ]:
ds = xr.open_dataset('../notebooks/data/SLP_ex.nc')
SLP = ds['SLP']
SLP_hPa = SLP / 100
lon = SLP_hPa['longitude'].values
lat = SLP_hPa['latitude'].values
lon2d, lat2d = np.meshgrid(lon, lat)
levels = np.arange(950, 1050, 10)

### Function to plot an animation over each time step in our data

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.contour(
        lon2d, lat2d, field,
        levels=levels,
        cmap=cmap,         
        linewidths=0.9,
        transform=ccrs.PlateCarree()
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Sea Level Pressure | time index = {t}")

:::{note}
The block of code below actually sets up and creates an animation of our desired data. Similar version of the blocks of code above and below this note will appear a few other times throughout this notebook. In the block of code below, we must call ".values" in the first line because SLP_hPa is an xarray dataarray, not a numpy array.
:::

In [ ]:
# Create map
vals = SLP_hPa.values
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("SLP")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

## Get our desired objects

Here is where we convert our xarray dataarray to a numpy array.

In [ ]:
SLP_arr = SLP_hPa.to_numpy()

### Choose a threshold for SLP

The threshold chosen here will serve as a dividing line between what is and is not considered a feeature of interest. In the block of code below, we will choose a threshold of 1000 hPa. This means that all of our features of interest must be comprised of grid cells whose SLP values are 1000 hPa or less. Other thresholds could be used too, such as departure from a long-term mean.

In [ ]:
SLP_thresh = 1000 #Units of hPa

SLP_1000 = np.where(SLP_arr < SLP_thresh, SLP, 0)

In [ ]:
s = generate_binary_structure(2,2)

In [ ]:
resolution = 1 #In degrees

labeled_arr = np.zeros_like(SLP_1000)

num_features_arr = np.zeros(len(SLP_1000))

for time_step in np.arange(0, len(SLP_1000)):
    labeled_SLP_field, num_features = label(SLP_1000[time_step], s)
  
    correct_labeled_SLP_field = crosses_dateline(labeled_SLP_field, resolution)
    
    num_features_arr[time_step] = num_features

    labeled_arr[time_step] = correct_labeled_SLP_field

    

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.pcolormesh(
        lon2d, lat2d, labeled_arr[t], shading = 'auto',
        cmap=cmap,         
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Identified Low-Pressure Objects | time index = {t}")

In [ ]:
# Create map
vals = labeled_arr
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("Object Number")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
#Get SLP minimum for each identified object

In [ ]:
minima = minimum_filter(SLP_arr, 5, mode=['nearest', 'wrap'], axes = [1, 2])
minima_3 = (SLP_arr == minima)
min_times, min_lats, min_lons = np.where(1 == minima_3)

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """

    where_time = np.where(min_times == t)

    proper_time_lats = min_lats[where_time]
    proper_time_lons = min_lons[where_time]
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.contourf(
        lon2d, lat2d, SLP_arr[t],
        cmap=cmap,         
        transform=ccrs.PlateCarree(),
    )

    cs = ax.scatter(
        lon[proper_time_lons], lat[proper_time_lats], 
        zorder = 2,
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"SLP and its Minima | time index = {t}")

In [ ]:
# Create map
vals = SLP_arr
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("Object Number")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
#Find the main SLP center for each object at each time step
deepest_mins_list = []

for time_step in np.arange(0, len(SLP_arr)):
    where_time = np.where(min_times == time_step)
    proper_time_lats = min_lats[where_time]
    proper_time_lons = min_lons[where_time]
    
    deepest_mins = {}
    
    for label_num in range(1, int(labeled_arr[time_step].max() + 1)):

        mask = (labeled_arr[time_step, proper_time_lats, proper_time_lons] == label_num)

        if(not np.any(mask)):
            continue

        obj_lat = proper_time_lats[mask]
        obj_lon = proper_time_lons[mask]

        obj_slp = SLP_arr[time_step][obj_lat, obj_lon]

        lowest = np.argmin(obj_slp)

        deepest_mins[label_num] = (lat[obj_lat[lowest]], obj_lon[lowest])
    deepest_mins_list.append(deepest_mins.copy())
    deepest_mins.clear()
    

In [ ]:
#We now have all minima at each time step. Now, it's time to stitch the tracks together
#Start with time step 1, check to see what's within 500 km. If none, then the track ends

first_time_step = deepest_mins_list[0]

track_no = 1
for min_point in first_time_step:
    print('----------------------------')
    print('Track Number ' + str(track_no))
    time_ind = 0
    
    #Start with minimum for first object
    min_point_lat = first_time_step[min_point][0]
    min_point_lon = first_time_step[min_point][1]
    print('Time step 1:', min_point_lat, min_point_lon)
    still_exists = True

    while(still_exists):
        time_ind += 1

        if(time_ind > len(deepest_mins_list) - 1):
            break
        
        #Go to next time step
        next_time_step = deepest_mins_list[time_ind]
        
        distances = np.zeros((len(next_time_step.items())))
        
        ind = 0
        for feat in next_time_step.values():
            dist = distance(min_point_lon, min_point_lat, feat[1], feat[0])
            distances[ind] = dist

            ind += 1
        
        min_dist = np.argmin(distances)
        #print(min_dist, next_time_step)
        if(min_dist < 500):
            new_min_point_lat = list(next_time_step.items())[min_dist][1][0]
            new_min_point_lon = list(next_time_step.items())[min_dist][1][1]
            print('Time step ' + str(time_ind + 1) + ':', new_min_point_lat, new_min_point_lon)
        else:
            still_exists = False
    track_no += 1
    